# 04 — Error analysis

Drills into the most confidently-wrong predictions from notebook 03, alongside the evidence sentence that led the classifier there — the fastest way to tell a parsing bug apart from a genuine model miss on the local sample set.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import pandas as pd

from report2label.extraction.label_mapper import LabelVocabulary
from report2label.utils.io import read_yaml
from report2label.validation.error_analysis import find_mismatches
from report2label.validation.evaluator import load_predictions

pipeline_config = read_yaml(PROJECT_ROOT / "configs" / "pipeline.yaml")
label_vocab = LabelVocabulary.from_yaml(PROJECT_ROOT / "configs" / "labels.yaml")

PREDICTIONS_DIR = PROJECT_ROOT / pipeline_config["paths"]["predictions_dir"]
ANNOTATIONS_PATH = PROJECT_ROOT / pipeline_config["paths"]["annotations_dir"] / "manual_labels.csv"

predictions = load_predictions(PREDICTIONS_DIR)
annotations = pd.read_csv(ANNOTATIONS_PATH, dtype={"ct_id": str})
mismatches = find_mismatches(predictions, annotations, label_vocab.names)

In [ ]:
for mismatch in mismatches[:20]:
    print(f"[{mismatch['kind']}] ct_id={mismatch['ct_id']} label={mismatch['label']} "
          f"true={mismatch['true']} pred={mismatch['predicted']} p={mismatch['probability']:.3f}")
    for item in mismatch["evidence"][:2]:
        print(f"    ({item['probability']:.3f}) {item['sentence']}")
    print()

In [ ]:
# Mismatch counts by label — which abnormalities need the most attention.
pd.DataFrame(mismatches).groupby(["label", "kind"]).size().unstack(fill_value=0)